# Δ peak-SWE timing and Δ SDD scatter comparison

This standalone notebook compares **Noah-MP** and **iSnobal** timing errors relative to **SNOTEL** at the three East River SNOTEL stations for **WY2018–WY2026**.

## Scientific inputs

The notebook **does not recalculate peak dates or snow disappearance dates from the daily SWE series**. It reads the validated workflow product:

`output/output1/tables/table4_snotel_annual_metrics.csv`

The v0.8.2 workflow creates that table with `calculate_snotel_metrics()`.

For each source and each station × water year:

- `peak_date_*` is the date of that source's own annual maximum SWE.
- `disappearance_date_*` is the existing **SWE-based** snow disappearance date from the workflow.

The delivered v0.8.2 implementation defines SWE disappearance as the first day of the first **7-day consecutive run with SWE < 10 mm**, searched after that source's own peak date. This notebook consumes the resulting validated dates rather than reimplementing that logic.

## Scatter definitions

Panel **(A)** — Δ peak-SWE timing:

- x = Noah-MP peak date − SNOTEL peak date (day)
- y = iSnobal peak date − SNOTEL peak date (day)

Panel **(B)** — Δ SDD:

- x = Noah-MP SDD − SNOTEL SDD (day)
- y = iSnobal SDD − SNOTEL SDD (day)

Positive values mean the model timing is **later** than SNOTEL; negative values mean **earlier**.

Each point is one station × water year. All stations use the same circular marker and are distinguished by color.

## Statistics

Two CSVs are written:

1. A station-year detail table containing exact dates, day-of-water-year (DOWY), signed differences, and peak SWE values.
2. A summary table containing `n`, Pearson `r`, `MB`, and `RMSE` for iSnobal and Noah-MP relative to SNOTEL:
   - separately for each station;
   - all stations combined.

For `r`, dates are converted to **day-of-water-year** before correlation. This avoids the trivial correlation that would result from correlating absolute calendar ordinals across successive years. `MB` and `RMSE` are calculated directly from signed model-minus-SNOTEL date differences in days.

WY2026 is intentionally partial in the workflow. If a validated SDD is unavailable by the analysis end date, that station-year is retained in the detail CSV with a missing SDD and is omitted only from calculations requiring that SDD.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

import numpy as np
import pandas as pd

try:
    from PIL import Image
except ImportError as exc:
    raise RuntimeError(
        "Pillow is required to guarantee RGB PNG export."
    ) from exc


# -------------------------------------------------------------------------
# East River workflow bootstrap
# -------------------------------------------------------------------------
REPO_ROOT = Path("/bsuscratch/zihaozhang/iSnobal/East_River_Workflow_Code_v0.8.2")
CONFIG_PATH = REPO_ROOT / "config" / "east_river_config.yaml"

try:
    from east_river_workflow.config import load_config
    from east_river_workflow.utils import water_year_bounds
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    from east_river_workflow.config import load_config
    from east_river_workflow.utils import water_year_bounds


In [ ]:
# -------------------------------------------------------------------------
# Current East River JHM/AMS plotting standard
# -------------------------------------------------------------------------
FONT_NAME = "Nimbus Sans"

available_font_names = {f.name for f in fm.fontManager.ttflist}
if FONT_NAME not in available_font_names:
    raise RuntimeError(
        "Nimbus Sans is required by the current East River plotting standard "
        "but is not available in this environment."
    )

# Resolve the actual font file with fallback disabled so the notebook cannot
# silently render with another font.
resolved_font_path = fm.findfont(
    fm.FontProperties(family=FONT_NAME),
    fallback_to_default=False,
)
print(f"Resolved publication font: {FONT_NAME}")
print(f"Resolved font file: {resolved_font_path}")

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [FONT_NAME],
    "font.size": 18,
    "axes.titlesize": 20,
    "axes.titleweight": "normal",
    "axes.labelweight": "normal",
    "font.weight": "normal",
    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# User-requested station encoding: same point symbol, different colors.
STATION_COLORS = {
    "Butte": "#0072B2",
    "Schofield Pass": "#D55E00",
    "Upper Taylor": "#009E73",
}
POINT_MARKER = "o"
POINT_SIZE = 90
POINT_EDGE_COLOR = "black"
POINT_EDGE_WIDTH = 0.6
POINT_ALPHA = 0.95

REFERENCE_LINE_COLOR = "0.35"
REFERENCE_LINE_WIDTH = 1.0
REFERENCE_LINE_STYLE = (0, (1.5, 2.5))

FIGSIZE = (14, 6.8)
EXPORT_DPI = 600


In [ ]:
# -------------------------------------------------------------------------
# Load the validated station-year timing fields from Table 4
# -------------------------------------------------------------------------
cfg = load_config(CONFIG_PATH)

TABLE4_PATH = cfg.output_dir / "tables" / "table4_snotel_annual_metrics.csv"

FIGURE_DIR = cfg.output_dir / "figures" / "publication"
TABLE_DIR = cfg.output_dir / "tables" / "publication"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_STEM = "delta_peak_swe_sdd_station_scatter"
DETAIL_CSV = TABLE_DIR / "delta_peak_swe_sdd_station_year_detail_wy2018_2026.csv"
STATS_CSV = TABLE_DIR / "delta_peak_swe_sdd_statistics_wy2018_2026.csv"

if not TABLE4_PATH.exists():
    raise FileNotFoundError(
        f"Validated Table 4 was not found: {TABLE4_PATH}\n"
        "Run the East River v0.8.2 workflow first so the validated station-year "
        "peak and disappearance dates exist."
    )

raw = pd.read_csv(TABLE4_PATH)

required_columns = [
    "station_id", "station_name", "water_year",
    "peak_SWE_SNOTEL_mm", "peak_SWE_I_mm", "peak_SWE_N_mm",
    "peak_date_SNOTEL", "peak_date_I", "peak_date_N",
    "disappearance_date_SNOTEL", "disappearance_date_I", "disappearance_date_N",
]
missing = [c for c in required_columns if c not in raw.columns]
if missing:
    raise ValueError(
        "Table 4 is missing required validated fields: "
        + ", ".join(missing)
    )

analysis_years = [int(y) for y in cfg.section("project")["water_years"]]
expected_years = list(range(2018, 2027))
if analysis_years != expected_years:
    raise ValueError(
        f"This notebook is defined for WY2018-WY2026, but config has {analysis_years}."
    )

raw["station_id"] = pd.to_numeric(raw["station_id"], errors="raise").astype(int)
raw["water_year"] = pd.to_numeric(raw["water_year"], errors="raise").astype(int)

expected_stations = [
    (int(s["id"]), str(s["name"]))
    for s in cfg.section("snotel")["stations"]
]

if raw.duplicated(["station_id", "water_year"]).any():
    dup = raw.loc[
        raw.duplicated(["station_id", "water_year"], keep=False),
        ["station_id", "station_name", "water_year"],
    ]
    raise ValueError(
        "Duplicate station-year rows found in Table 4:\n"
        + dup.to_string(index=False)
    )

expected_keys = {
    (sid, wy)
    for sid, _ in expected_stations
    for wy in expected_years
}
actual_keys = set(
    zip(raw["station_id"].astype(int), raw["water_year"].astype(int))
)
missing_keys = sorted(expected_keys - actual_keys)
if missing_keys:
    raise ValueError(
        "Table 4 is missing expected station-year rows: "
        f"{missing_keys}"
    )

print("Validated input:", TABLE4_PATH)
print("Rows:", len(raw))
print("Stations:", [name for _, name in expected_stations])
print("Water years:", expected_years)

# Report the threshold configuration used upstream.
h2_cfg = cfg.section("h2")
print(
    "Upstream SWE-disappearance configuration:",
    f"SWE < {float(h2_cfg['disappearance_threshold_swe_mm']):g} mm",
    f"for {int(h2_cfg['disappearance_consecutive_days'])} consecutive days",
)


In [ ]:
# -------------------------------------------------------------------------
# Construct the station-year detail table from validated dates
# -------------------------------------------------------------------------
DATE_COLUMNS = [
    "peak_date_SNOTEL", "peak_date_I", "peak_date_N",
    "disappearance_date_SNOTEL", "disappearance_date_I", "disappearance_date_N",
]
for col in DATE_COLUMNS:
    raw[col] = pd.to_datetime(raw[col], errors="coerce").dt.normalize()


def date_difference_days(model_date: pd.Series, reference_date: pd.Series) -> pd.Series:
    """Signed model-minus-reference date difference in days."""
    return (model_date - reference_date).dt.days.astype("Float64")


def date_to_dowy(date_value, water_year: int) -> float:
    """Convert a calendar date to day-of-water-year, with Oct 1 = 1."""
    if pd.isna(date_value):
        return np.nan
    start, _ = water_year_bounds(int(water_year))
    return float(
        (pd.Timestamp(date_value).normalize() - pd.Timestamp(start).normalize()).days + 1
    )


detail = raw[
    [
        "station_id", "station_name", "water_year",
        "peak_SWE_SNOTEL_mm", "peak_SWE_I_mm", "peak_SWE_N_mm",
        "peak_date_SNOTEL", "peak_date_I", "peak_date_N",
        "disappearance_date_SNOTEL", "disappearance_date_I", "disappearance_date_N",
    ]
].copy()

detail = detail.rename(columns={
    "peak_SWE_I_mm": "peak_SWE_iSnobal_mm",
    "peak_SWE_N_mm": "peak_SWE_NoahMP_mm",
    "peak_date_I": "peak_date_iSnobal",
    "peak_date_N": "peak_date_NoahMP",
    "disappearance_date_SNOTEL": "SDD_SNOTEL",
    "disappearance_date_I": "SDD_iSnobal",
    "disappearance_date_N": "SDD_NoahMP",
})

detail["delta_peak_SWE_timing_iSnobal_vs_SNOTEL_days"] = date_difference_days(
    detail["peak_date_iSnobal"], detail["peak_date_SNOTEL"]
)
detail["delta_peak_SWE_timing_NoahMP_vs_SNOTEL_days"] = date_difference_days(
    detail["peak_date_NoahMP"], detail["peak_date_SNOTEL"]
)
detail["delta_SDD_iSnobal_vs_SNOTEL_days"] = date_difference_days(
    detail["SDD_iSnobal"], detail["SDD_SNOTEL"]
)
detail["delta_SDD_NoahMP_vs_SNOTEL_days"] = date_difference_days(
    detail["SDD_NoahMP"], detail["SDD_SNOTEL"]
)

for base in ["peak_date", "SDD"]:
    for source in ["SNOTEL", "iSnobal", "NoahMP"]:
        date_col = f"{base}_{source}"
        dowy_col = f"{base}_DOWY_{source}"
        detail[dowy_col] = [
            date_to_dowy(d, wy)
            for d, wy in zip(detail[date_col], detail["water_year"])
        ]

detail_columns = [
    "station_id", "station_name", "water_year",
    "peak_SWE_SNOTEL_mm", "peak_SWE_iSnobal_mm", "peak_SWE_NoahMP_mm",
    "peak_date_SNOTEL", "peak_date_iSnobal", "peak_date_NoahMP",
    "peak_date_DOWY_SNOTEL", "peak_date_DOWY_iSnobal", "peak_date_DOWY_NoahMP",
    "delta_peak_SWE_timing_iSnobal_vs_SNOTEL_days",
    "delta_peak_SWE_timing_NoahMP_vs_SNOTEL_days",
    "SDD_SNOTEL", "SDD_iSnobal", "SDD_NoahMP",
    "SDD_DOWY_SNOTEL", "SDD_DOWY_iSnobal", "SDD_DOWY_NoahMP",
    "delta_SDD_iSnobal_vs_SNOTEL_days",
    "delta_SDD_NoahMP_vs_SNOTEL_days",
]
detail = detail[detail_columns].sort_values(
    ["station_id", "water_year"]
).reset_index(drop=True)

display(detail)


In [ ]:
# -------------------------------------------------------------------------
# Summary statistics: per station and all stations combined
# -------------------------------------------------------------------------
def timing_stats(frame: pd.DataFrame, metric: str, model: str) -> dict:
    """
    r: Pearson correlation of model vs SNOTEL day-of-water-year.
    MB: mean(model_date - SNOTEL_date), days.
    RMSE: sqrt(mean((model_date - SNOTEL_date)^2)), days.
    """
    if metric == "peak_SWE_timing":
        ref_dowy = "peak_date_DOWY_SNOTEL"
        model_dowy = f"peak_date_DOWY_{model}"
        diff_col = f"delta_peak_SWE_timing_{model}_vs_SNOTEL_days"
    elif metric == "SWE_SDD":
        ref_dowy = "SDD_DOWY_SNOTEL"
        model_dowy = f"SDD_DOWY_{model}"
        diff_col = f"delta_SDD_{model}_vs_SNOTEL_days"
    else:
        raise ValueError(metric)

    x = pd.to_numeric(frame[ref_dowy], errors="coerce")
    y = pd.to_numeric(frame[model_dowy], errors="coerce")
    d = pd.to_numeric(frame[diff_col], errors="coerce")

    valid = x.notna() & y.notna() & d.notna()
    xv = x[valid].to_numpy(dtype=float)
    yv = y[valid].to_numpy(dtype=float)
    dv = d[valid].to_numpy(dtype=float)
    n = len(dv)

    if n < 2 or np.std(xv) == 0 or np.std(yv) == 0:
        r = np.nan
    else:
        r = float(np.corrcoef(xv, yv)[0, 1])

    mb = float(np.mean(dv)) if n else np.nan
    rmse = float(np.sqrt(np.mean(dv ** 2))) if n else np.nan

    return {
        "n": int(n),
        "pearson_r": r,
        "MB_days": mb,
        "RMSE_days": rmse,
    }


stats_rows = []

for metric in ["peak_SWE_timing", "SWE_SDD"]:
    for model in ["iSnobal", "NoahMP"]:
        for sid, station_name in expected_stations:
            subset = detail[detail["station_id"].eq(sid)]
            values = timing_stats(subset, metric, model)
            stats_rows.append({
                "metric": metric,
                "scope": "station",
                "station_id": sid,
                "station_name": station_name,
                "model": "Noah-MP" if model == "NoahMP" else "iSnobal",
                "reference": "SNOTEL",
                "water_year_start": 2018,
                "water_year_end": 2026,
                "r_basis": "day_of_water_year",
                **values,
            })

        values = timing_stats(detail, metric, model)
        stats_rows.append({
            "metric": metric,
            "scope": "all_stations",
            "station_id": pd.NA,
            "station_name": "All stations",
            "model": "Noah-MP" if model == "NoahMP" else "iSnobal",
            "reference": "SNOTEL",
            "water_year_start": 2018,
            "water_year_end": 2026,
            "r_basis": "day_of_water_year",
            **values,
        })

stats = pd.DataFrame(stats_rows)
display(stats)


In [ ]:
# -------------------------------------------------------------------------
# Save the requested statistical products
# -------------------------------------------------------------------------
detail.to_csv(
    DETAIL_CSV,
    index=False,
    date_format="%Y-%m-%d",
    float_format="%.6g",
)
stats.to_csv(
    STATS_CSV,
    index=False,
    float_format="%.6g",
)

print("Saved detail CSV:", DETAIL_CSV)
print("Saved statistics CSV:", STATS_CSV)

panel_specs = {
    "(a) peak-SWE timing": (
        "delta_peak_SWE_timing_NoahMP_vs_SNOTEL_days",
        "delta_peak_SWE_timing_iSnobal_vs_SNOTEL_days",
    ),
    "(b) SWE SDD": (
        "delta_SDD_NoahMP_vs_SNOTEL_days",
        "delta_SDD_iSnobal_vs_SNOTEL_days",
    ),
}

for label, (xcol, ycol) in panel_specs.items():
    valid = detail[xcol].notna() & detail[ycol].notna()
    omitted = detail.loc[
        ~valid, ["station_name", "water_year", xcol, ycol]
    ]
    print(f"\n{label}: {int(valid.sum())} plotted station-year points")
    if not omitted.empty:
        print("Omitted because one or both plotted differences are missing:")
        print(omitted.to_string(index=False))


In [ ]:
# -------------------------------------------------------------------------
# Two-panel scatter figure
# -------------------------------------------------------------------------
def symmetric_limit(x, y, step: int = 10, minimum: int = 20) -> float:
    vals = np.concatenate([
        pd.to_numeric(x, errors="coerce").to_numpy(dtype=float),
        pd.to_numeric(y, errors="coerce").to_numpy(dtype=float),
    ])
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return float(minimum)
    raw_limit = max(float(np.max(np.abs(vals))) * 1.10, float(minimum))
    return float(np.ceil(raw_limit / step) * step)


def style_scatter_axis(ax, limit: float):
    ax.axhline(
        0,
        color=REFERENCE_LINE_COLOR,
        linewidth=REFERENCE_LINE_WIDTH,
        linestyle=REFERENCE_LINE_STYLE,
        zorder=1,
    )
    ax.axvline(
        0,
        color=REFERENCE_LINE_COLOR,
        linewidth=REFERENCE_LINE_WIDTH,
        linestyle=REFERENCE_LINE_STYLE,
        zorder=1,
    )

    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_aspect("equal", adjustable="box")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, symmetric=True, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5, symmetric=True, integer=True))

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(axis="both", which="major", width=1.0, length=5)


fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, squeeze=False)
axes = axes[0]

plot_specs = [
    {
        "panel": "(a)",
        "x": "delta_peak_SWE_timing_NoahMP_vs_SNOTEL_days",
        "y": "delta_peak_SWE_timing_iSnobal_vs_SNOTEL_days",
    },
    {
        "panel": "(b)",
        "x": "delta_SDD_NoahMP_vs_SNOTEL_days",
        "y": "delta_SDD_iSnobal_vs_SNOTEL_days",
    },
]

for ax, spec in zip(axes, plot_specs):
    xcol = spec["x"]
    ycol = spec["y"]

    limit = symmetric_limit(detail[xcol], detail[ycol])

    for sid, station_name in expected_stations:
        ss = detail[
            detail["station_id"].eq(sid)
            & detail[xcol].notna()
            & detail[ycol].notna()
        ]

        if station_name not in STATION_COLORS:
            raise KeyError(
                f"No station color defined for {station_name!r}. "
                "Update STATION_COLORS before plotting."
            )

        ax.scatter(
            ss[xcol],
            ss[ycol],
            marker=POINT_MARKER,
            s=POINT_SIZE,
            facecolors=STATION_COLORS[station_name],
            edgecolors=POINT_EDGE_COLOR,
            linewidths=POINT_EDGE_WIDTH,
            alpha=POINT_ALPHA,
            zorder=4,
        )

    style_scatter_axis(ax, limit)

    # Panel labels only; these are not panel titles.
    if spec["panel"] == "(a)":
        ax.set_title(
            "Δ SWEpeak",
            fontsize=20,
            fontweight="normal",
            pad=8,
        )
    else:
        ax.set_title(
            "Δ SDD",
            fontsize=20,
            fontweight="normal",
            pad=8,
        )

legend_handles = [
    Line2D(
        [], [],
        linestyle="None",
        marker=POINT_MARKER,
        markersize=np.sqrt(POINT_SIZE),
        markerfacecolor=STATION_COLORS[station_name],
        markeredgecolor=POINT_EDGE_COLOR,
        markeredgewidth=POINT_EDGE_WIDTH,
        label=station_name,
    )
    for _, station_name in expected_stations
]

# Plotting-protocol placement: boxed legend INSIDE the first subplot.
# It is shifted slightly downward to keep the (A) panel label unobstructed.
axes[0].legend(
    handles=legend_handles,
    loc="upper left",
    bbox_to_anchor=(0.02, 0.89),
    ncol=1,
    frameon=True,
    facecolor="white",
    edgecolor="0.4",
    framealpha=0.9,
    fancybox=False,
    borderpad=0.5,
    handlelength=1.5,
)


# Explicitly center the shared labels in figure coordinates.
fig.supxlabel(
    "Noah-MP − SNOTEL (day)",
    x=0.5,
    y=0.055,
    ha="center",
    va="center",
)
fig.supylabel(
    "iSnobal − SNOTEL (day)",
    x=0.05,
    y=0.5,
    ha="center",
    va="center",
)

fig.subplots_adjust(
    left=0.105,
    right=0.98,
    bottom=0.15,
    top=0.96,
    wspace=0.20,
)

png_path = FIGURE_DIR / f"{FIGURE_STEM}.png"
pdf_path = FIGURE_DIR / f"{FIGURE_STEM}.pdf"

fig.savefig(
    png_path,
    dpi=EXPORT_DPI,
    bbox_inches="tight",
    pad_inches=0.05,
    facecolor="white",
    transparent=False,
)
fig.savefig(
    pdf_path,
    bbox_inches="tight",
    pad_inches=0.05,
    facecolor="white",
    transparent=False,
)

with Image.open(png_path) as im:
    if im.mode != "RGB":
        rgb = im.convert("RGB")
        rgb.save(png_path, dpi=(EXPORT_DPI, EXPORT_DPI))

print("Saved figure:", png_path)
print("Saved figure:", pdf_path)

plt.show()


## Outputs

The notebook writes:

**Figure**
- `output/output1/figures/publication/delta_peak_swe_sdd_station_scatter.png`
- `output/output1/figures/publication/delta_peak_swe_sdd_station_scatter.pdf`

**Detailed station-year timing table**
- `output/output1/tables/publication/delta_peak_swe_sdd_station_year_detail_wy2018_2026.csv`

**Per-station + combined statistics**
- `output/output1/tables/publication/delta_peak_swe_sdd_statistics_wy2018_2026.csv`

The figure contains no statistical annotations and no panel titles. Only the requested panel labels **(A)** and **(B)** are shown.

Plotting-layout notes for this revision:
- Nimbus Sans is resolved with fallback disabled and its resolved font file is printed.
- The station legend is boxed inside panel (A), following the project plotting protocol.
- The shared y-axis label is explicitly centered at `y=0.5`.
